# Phase 7 — Test Set Inference, Submission Generation & Validation
### Amazon ML Challenge: Multi-Source Entity Resolution

**Objective:**
1. Generate high-precision candidate sets for the **Test Set** (`test_source1.tsv` vs `test_source2/3`), fully supporting **France 🇫🇷**, **US 🇺🇸**, and **India 🇮🇳**.
2. Score candidate pairs using the trained **Blended Random Forest + XGBoost Ensemble**.
3. Apply **$F_{0.5}$-Tuned Margin Thresholding** ($	au^*$) to maximize precision and preserve singletons (scoring 1.0).
4. Format outputs strictly to the competition schema:
   - `output/matching_results.tsv` (Leaderboard predictions)
   - `output/candidate_pairs.tsv` (Test candidate pool)
5. Run `utils/validate_submission.py` to guarantee 100% compliance.
6. Package into final `<team_name>_submission.zip`.

In [1]:
from __future__ import annotations

import csv
import re
import sys
import time
import unicodedata
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple

import joblib
import numpy as np

# Auto-detect project paths
if Path("/content/AMAZON_ML_CHALLENGE-main").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE-main")
elif Path("/content/AMAZON_ML_CHALLENGE").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE")
elif Path("..").resolve().name == "AMAZON_ML_CHALLENGE":
    ROOT = Path("..").resolve()
else:
    ROOT = Path(".").resolve()

TEST_DIR = ROOT / "student_resource" / "dataset" / "test"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = ROOT / "data" / "processed" / "models"

MATCHING_OUT = OUTPUT_DIR / "matching_results.tsv"
CANDIDATES_OUT = OUTPUT_DIR / "candidate_pairs.tsv"

print(f"Project Root: {ROOT}")
print(f"Test Directory: {TEST_DIR} (Exists: {TEST_DIR.exists()})")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Target Matching File: {MATCHING_OUT}")
print(f"Target Candidate File: {CANDIDATES_OUT}")

## Step 1: Load Trained Ensemble & Blending Config

In [2]:
rf_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")
xgb_model = joblib.load(MODELS_DIR / "xgboost_model.joblib")
config = joblib.load(MODELS_DIR / "ensemble_config.joblib")

print("Loaded Trained Ensemble:")
print(f"- XGBoost Weight (w1) : {config['w_xgb']:.2f}")
print(f"- Random Forest Weight: {config['w_rf']:.2f}")
print(f"- Optimal Threshold (τ): {config['threshold']}")

## Step 2: Test Set Blocking & Inference Pipeline
Streams `test_source1.tsv` queries, extracts top candidates, computes the 12 features, blends RF + XGBoost, and generates both `matching_results.tsv` and `candidate_pairs.tsv`.

In [3]:
# File definitions
TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

print("Ready for full test inference pipeline.")

## Step 3: Submission Verification with `validate_submission.py`

In [4]:
import subprocess

validator_script = ROOT / "student_resource" / "utils" / "validate_submission.py"
cmd = [
    sys.executable,
    str(validator_script),
    "--matching", str(MATCHING_OUT),
    "--candidate", str(CANDIDATES_OUT),
    "--test-dir", str(TEST_DIR)
]
print("Running Official Validator...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

if result.returncode == 0:
    print("🎉 VALIDATION PASSED! Submissions are 100% compliant with competition rules!")
else:
    print("❌ VALIDATION ISSUES DETECTED. Please review the output above.")